# 대규모 합성 분개장 생성

가상 자동차부품 제조기업의 거래 규칙과 마스터 데이터를 기반으로
정상 거래와 오류 거래가 포함된 500건의 합성 분개장을 생성한다.

## 1. 라이브러리와 데이터 생성 조건 설정

합성 데이터 생성에 필요한 라이브러리와 마스터 데이터를 불러온다.
재현 가능한 결과를 위해 난수 기준값과 생성할 거래 수를 설정한다.

In [1]:
# 표 형태의 데이터 처리
import pandas as pd

# 무작위 날짜, 금액, 거래 유형 생성
import numpy as np

# 프로젝트 파일 경로 관리
from pathlib import Path


# 현재 실행 위치를 기준으로 프로젝트 루트 설정
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

# 마스터 데이터 파일 경로
accounts_path = (
    project_root
    / "data"
    / "master"
    / "accounts.csv"
)

vendors_path = (
    project_root
    / "data"
    / "master"
    / "vendors.csv"
)

departments_path = (
    project_root
    / "data"
    / "master"
    / "departments.csv"
)

# 계정과목·거래처·부서 기준정보 불러오기
accounts = pd.read_csv(accounts_path)
vendors = pd.read_csv(vendors_path)
departments = pd.read_csv(departments_path)

# 코드를 실행할 때마다 같은 데이터가 생성되도록 난수 기준값 고정
random_seed = 42
np.random.seed(random_seed)

# 생성할 전체 거래 수
total_transaction_count = 500

# 전체 거래 중 오류를 삽입할 비율
error_rate = 0.10

# 정상 거래와 오류 거래의 목표 건수 계산
error_transaction_count = int(
    total_transaction_count * error_rate
)

normal_transaction_count = (
    total_transaction_count
    - error_transaction_count
)

print("난수 기준값:", random_seed)
print("전체 생성 예정 거래:", total_transaction_count)
print("정상 거래 예정:", normal_transaction_count)
print("오류 거래 예정:", error_transaction_count)

print()
print("계정과목 기준정보:", len(accounts))
print("거래처 기준정보:", len(vendors))
print("부서 기준정보:", len(departments))

난수 기준값: 42
전체 생성 예정 거래: 500
정상 거래 예정: 450
오류 거래 예정: 50

계정과목 기준정보: 22
거래처 기준정보: 12
부서 기준정보: 7


In [2]:
# 표 형태의 데이터 처리
import pandas as pd

# 무작위 날짜, 금액, 거래 유형 생성
import numpy as np

# 프로젝트 파일 경로 관리
from pathlib import Path


# 현재 실행 위치를 기준으로 프로젝트 루트 설정
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

# 마스터 데이터 파일 경로
accounts_path = (
    project_root
    / "data"
    / "master"
    / "accounts.csv"
)

vendors_path = (
    project_root
    / "data"
    / "master"
    / "vendors.csv"
)

departments_path = (
    project_root
    / "data"
    / "master"
    / "departments.csv"
)

# 계정과목·거래처·부서 기준정보 불러오기
accounts = pd.read_csv(accounts_path)
vendors = pd.read_csv(vendors_path)
departments = pd.read_csv(departments_path)

# 코드를 실행할 때마다 같은 데이터가 생성되도록 난수 기준값 고정
random_seed = 42
np.random.seed(random_seed)

# 생성할 전체 거래 수
total_transaction_count = 500

# 전체 거래 중 오류를 삽입할 비율
error_rate = 0.10

# 정상 거래와 오류 거래의 목표 건수 계산
error_transaction_count = int(
    total_transaction_count * error_rate
)

normal_transaction_count = (
    total_transaction_count
    - error_transaction_count
)

print("난수 기준값:", random_seed)
print("전체 생성 예정 거래:", total_transaction_count)
print("정상 거래 예정:", normal_transaction_count)
print("오류 거래 예정:", error_transaction_count)

print()
print("계정과목 기준정보:", len(accounts))
print("거래처 기준정보:", len(vendors))
print("부서 기준정보:", len(departments))

난수 기준값: 42
전체 생성 예정 거래: 500
정상 거래 예정: 450
오류 거래 예정: 50

계정과목 기준정보: 22
거래처 기준정보: 12
부서 기준정보: 7


## 2. 제조기업 거래 유형별 생성 규칙 정의

원재료 매입, 소모품 구매, 운송비, 전기요금, 설비 구매,
경영서비스, 제품 판매의 거래 비중과 금액 범위를 정의한다.

In [3]:
# 합성 분개장을 생성할 때 사용할 거래 유형별 업무 규칙
transaction_rules = [
    {
        "transaction_type": "원재료 매입",
        "partner_codes": ["V001", "V002"],
        "department_codes": ["D002"],
        "descriptions": [
            "고무 원재료 외상 매입",
            "합성고무 원재료 구매",
            "자동차부품용 소재 매입"
        ],
        "supply_min": 1_000_000,
        "supply_max": 8_000_000,
        "evidence_type": "전자세금계산서",
        "transaction_direction": "매입",
        "selection_weight": 0.30
    },
    {
        "transaction_type": "산업용 소모품 구매",
        "partner_codes": ["V003", "V004"],
        "department_codes": [
            "D003",
            "D004",
            "D005",
            "D007"
        ],
        "descriptions": [
            "생산 작업용 장갑 구매",
            "검사용 소모품 구매",
            "제품 포장재 구매",
            "현장 안전용품 구매"
        ],
        "supply_min": 100_000,
        "supply_max": 1_000_000,
        "evidence_type": "법인카드",
        "transaction_direction": "매입",
        "selection_weight": 0.15
    },
    {
        "transaction_type": "운송비",
        "partner_codes": ["V005"],
        "department_codes": ["D007"],
        "descriptions": [
            "제품 납품 운송비",
            "원재료 입고 운송비",
            "긴급 납품 운송비"
        ],
        "supply_min": 200_000,
        "supply_max": 1_500_000,
        "evidence_type": "전자세금계산서",
        "transaction_direction": "매입",
        "selection_weight": 0.10
    },
    {
        "transaction_type": "공장 전기요금",
        "partner_codes": ["V006"],
        "department_codes": [
            "D003",
            "D004"
        ],
        "descriptions": [
            "생산1공장 전기요금",
            "생산2공장 전기요금",
            "공장 설비 전력 사용료"
        ],
        "supply_min": 1_000_000,
        "supply_max": 5_000_000,
        "evidence_type": "전자세금계산서",
        "transaction_direction": "매입",
        "selection_weight": 0.10
    },
    {
        "transaction_type": "생산설비 구매",
        "partner_codes": ["V007"],
        "department_codes": ["D002"],
        "descriptions": [
            "생산설비 구매",
            "품질검사 설비 구매",
            "공정 자동화 장비 구매"
        ],
        "supply_min": 10_000_000,
        "supply_max": 50_000_000,
        "evidence_type": "전자세금계산서",
        "transaction_direction": "매입",
        "selection_weight": 0.05
    },
    {
        "transaction_type": "경영서비스 수수료",
        "partner_codes": ["V008"],
        "department_codes": ["D001"],
        "descriptions": [
            "회계 자문 수수료",
            "경영관리 서비스 비용",
            "업무 시스템 자문료"
        ],
        "supply_min": 500_000,
        "supply_max": 3_000_000,
        "evidence_type": "전자세금계산서",
        "transaction_direction": "매입",
        "selection_weight": 0.05
    },
    {
        "transaction_type": "제품 판매",
        "partner_codes": [
            "C001",
            "C002",
            "C003",
            "C004"
        ],
        "department_codes": ["D006"],
        "descriptions": [
            "자동차용 고무부품 외상 판매",
            "자동차용 씰링부품 외상 판매",
            "방진부품 외상 판매",
            "고무호스 제품 외상 판매"
        ],
        "supply_min": 3_000_000,
        "supply_max": 20_000_000,
        "evidence_type": "전자세금계산서",
        "transaction_direction": "매출",
        "selection_weight": 0.25
    }
]

# 거래 유형별 선택 확률의 합계 계산
total_weight = sum(
    rule["selection_weight"]
    for rule in transaction_rules
)

print("정의한 거래 유형 수:", len(transaction_rules))
print("거래 유형 선택 확률 합계:", total_weight)

# 거래 유형과 선택 비중 확인
rule_summary = pd.DataFrame(
    [
        {
            "transaction_type": rule["transaction_type"],
            "direction": rule["transaction_direction"],
            "minimum_amount": rule["supply_min"],
            "maximum_amount": rule["supply_max"],
            "selection_weight": rule["selection_weight"]
        }
        for rule in transaction_rules
    ]
)

rule_summary

정의한 거래 유형 수: 7
거래 유형 선택 확률 합계: 1.0


,transaction_type,direction,minimum_amount,maximum_amount,selection_weight
0,원재료 매입,매입,1000000,8000000,0.30
1,산업용 소모품 구매,매입,100000,1000000,0.15
2,운송비,매입,200000,1500000,0.10
3,공장 전기요금,매입,1000000,5000000,0.10
4,생산설비 구매,매입,10000000,50000000,0.05
5,경영서비스 수수료,매입,500000,3000000,0.05
6,제품 판매,매출,3000000,20000000,0.25


## 3. 거래 규칙을 적용한 정상 분개장 생성

거래 유형별 비중, 거래처, 부서, 금액 범위를 이용해
2026년 8월 정상 분개장 500건을 생성한다.

In [4]:
# 거래 유형 선택에 사용할 확률 목록
selection_weights = [
    rule["selection_weight"]
    for rule in transaction_rules
]

# 생성한 정상 거래를 저장할 빈 목록
generated_transactions = []


# 설정한 전체 거래 수만큼 반복
for transaction_number in range(
    1,
    total_transaction_count + 1
):
    # 거래 유형별 확률을 반영해 규칙 하나 선택
    rule_index = np.random.choice(
        len(transaction_rules),
        p=selection_weights
    )

    selected_rule = transaction_rules[rule_index]

    # 선택된 거래 유형에 맞는 거래처와 부서 선택
    partner_code = np.random.choice(
        selected_rule["partner_codes"]
    )

    department_code = np.random.choice(
        selected_rule["department_codes"]
    )

    description = np.random.choice(
        selected_rule["descriptions"]
    )

    # 거래처 기준표에서 선택된 거래처 정보 조회
    partner_record = vendors.loc[
        vendors["partner_code"] == partner_code
    ].iloc[0]

    # 2026년 8월 1일부터 31일 사이의 날짜 생성
    random_day = np.random.randint(0, 31)

    transaction_date = (
        pd.Timestamp("2026-08-01")
        + pd.Timedelta(days=int(random_day))
    )

    # 거래 유형의 금액 범위 안에서 만 원 단위 공급가액 생성
    minimum_unit = (
        selected_rule["supply_min"] // 10_000
    )

    maximum_unit = (
        selected_rule["supply_max"] // 10_000
    )

    supply_amount = (
        np.random.randint(
            minimum_unit,
            maximum_unit + 1
        )
        * 10_000
    )

    # 일반 과세 거래를 가정해 공급가액의 10%로 부가세 계산
    vat_amount = round(supply_amount * 0.1)

    # 공급가액과 부가세를 더해 총금액 계산
    total_amount = supply_amount + vat_amount

    # 거래처의 기본 계정과 정산 계정 가져오기
    default_account_code = int(
        partner_record["default_account_code"]
    )

    settlement_account_code = int(
        partner_record["settlement_account_code"]
    )

    # 거래처의 결제 방식 확인
    payment_method = partner_record["payment_method"]

    # 법인카드 거래는 카드 증빙으로 처리
    if payment_method == "법인카드":
        evidence_type = "법인카드"
        evidence_prefix = "CARD"
    else:
        evidence_type = selected_rule["evidence_type"]
        evidence_prefix = "TAX"

    # 거래마다 중복되지 않는 증빙번호 생성
    evidence_no = (
        f"{evidence_prefix}-202608-"
        f"{transaction_number:04d}"
    )

    # 매입 거래의 차변·대변 계정 설정
    if selected_rule["transaction_direction"] == "매입":
        debit_account_1 = default_account_code
        debit_amount_1 = supply_amount

        debit_account_2 = 1180
        debit_amount_2 = vat_amount

        credit_account_1 = settlement_account_code
        credit_amount_1 = total_amount

        credit_account_2 = None
        credit_amount_2 = 0

    # 매출 거래의 차변·대변 계정 설정
    else:
        debit_account_1 = settlement_account_code
        debit_amount_1 = total_amount

        debit_account_2 = None
        debit_amount_2 = 0

        credit_account_1 = default_account_code
        credit_amount_1 = supply_amount

        credit_account_2 = 2180
        credit_amount_2 = vat_amount

    # 완성된 거래 한 건을 목록에 추가
    generated_transactions.append(
        {
            "voucher_id": (
                f"JV202608"
                f"{transaction_number + 1000:04d}"
            ),
            "transaction_date": transaction_date,
            "transaction_type": selected_rule[
                "transaction_type"
            ],
            "department_code": department_code,
            "partner_code": partner_code,
            "evidence_type": evidence_type,
            "evidence_no": evidence_no,
            "description": description,
            "debit_account_1": debit_account_1,
            "debit_amount_1": debit_amount_1,
            "debit_account_2": debit_account_2,
            "debit_amount_2": debit_amount_2,
            "credit_account_1": credit_account_1,
            "credit_amount_1": credit_amount_1,
            "credit_account_2": credit_account_2,
            "credit_amount_2": credit_amount_2,
            "supply_amount": supply_amount,
            "vat_amount": vat_amount,
            "total_amount": total_amount,
            "remarks": "포트폴리오용 합성 거래"
        }
    )


# 생성한 거래 목록을 데이터프레임으로 변환
bulk_normal_journal = pd.DataFrame(
    generated_transactions
)

# 생성 결과 확인
print("생성된 정상 거래 수:", len(bulk_normal_journal))
print(
    "고유 전표번호 수:",
    bulk_normal_journal["voucher_id"].nunique()
)
print(
    "최초 거래일:",
    bulk_normal_journal["transaction_date"].min().date()
)
print(
    "최종 거래일:",
    bulk_normal_journal["transaction_date"].max().date()
)

# 거래 유형별 실제 생성 건수 확인
transaction_type_summary = (
    bulk_normal_journal["transaction_type"]
    .value_counts()
    .rename_axis("transaction_type")
    .reset_index(name="transaction_count")
)

transaction_type_summary

생성된 정상 거래 수: 500
고유 전표번호 수: 500
최초 거래일: 2026-08-01
최종 거래일: 2026-08-31


,transaction_type,transaction_count
0,원재료 매입,146
1,제품 판매,114
2,운송비,67
3,산업용 소모품 구매,65
4,공장 전기요금,61
5,생산설비 구매,25
6,경영서비스 수수료,22


## 4. 생성된 정상 분개장의 품질 검증

오류를 삽입하기 전에 생성된 500건의 차변·대변, 부가세,
계정과목·거래처·부서 코드, 전표번호와 증빙번호를 검사한다.

In [5]:
# 유효한 계정과목·거래처·부서 코드 집합 생성
valid_account_codes = set(
    accounts["account_code"]
    .dropna()
    .astype(int)
)

valid_partner_codes = set(
    vendors.loc[
        vendors["is_active"] == "Y",
        "partner_code"
    ].astype(str)
)

valid_department_codes = set(
    departments.loc[
        departments["is_active"] == "Y",
        "department_code"
    ].astype(str)
)


# 거래별 차변 합계 계산
bulk_normal_journal["debit_total_check"] = (
    bulk_normal_journal["debit_amount_1"].fillna(0)
    + bulk_normal_journal["debit_amount_2"].fillna(0)
)

# 거래별 대변 합계 계산
bulk_normal_journal["credit_total_check"] = (
    bulk_normal_journal["credit_amount_1"].fillna(0)
    + bulk_normal_journal["credit_amount_2"].fillna(0)
)

# 차변 합계와 대변 합계의 일치 여부
bulk_normal_journal["balance_valid"] = (
    bulk_normal_journal["debit_total_check"]
    == bulk_normal_journal["credit_total_check"]
)

# 전표 총금액과 분개 합계의 일치 여부
bulk_normal_journal["total_amount_valid"] = (
    (
        bulk_normal_journal["total_amount"]
        == bulk_normal_journal["debit_total_check"]
    )
    & (
        bulk_normal_journal["total_amount"]
        == bulk_normal_journal["credit_total_check"]
    )
)

# 공급가액을 기준으로 예상 부가세 계산
expected_vat = (
    bulk_normal_journal["supply_amount"]
    * 0.1
).round()

# 입력 부가세가 예상 부가세와 일치하는지 확인
bulk_normal_journal["vat_valid"] = (
    bulk_normal_journal["vat_amount"]
    == expected_vat
)

# 거래처 코드와 부서 코드의 유효성 확인
bulk_normal_journal["partner_valid"] = (
    bulk_normal_journal["partner_code"]
    .isin(valid_partner_codes)
)

bulk_normal_journal["department_valid"] = (
    bulk_normal_journal["department_code"]
    .isin(valid_department_codes)
)


# 한 거래에 입력된 모든 계정코드의 유효성을 확인하는 함수
def validate_account_codes(row):
    account_columns = [
        "debit_account_1",
        "debit_account_2",
        "credit_account_1",
        "credit_account_2"
    ]

    for column in account_columns:
        account_value = row[column]

        # 사용하지 않는 두 번째 계정의 빈칸은 정상으로 처리
        if pd.isna(account_value):
            continue

        if int(account_value) not in valid_account_codes:
            return False

    return True


# 각 거래에 계정코드 검증 함수 적용
bulk_normal_journal["account_valid"] = (
    bulk_normal_journal.apply(
        validate_account_codes,
        axis=1
    )
)

# 거래일자가 2026년 8월에 포함되는지 확인
bulk_normal_journal["date_valid"] = (
    (
        bulk_normal_journal["transaction_date"]
        >= pd.Timestamp("2026-08-01")
    )
    & (
        bulk_normal_journal["transaction_date"]
        < pd.Timestamp("2026-09-01")
    )
)

# 필수 텍스트값의 누락 여부 확인
required_columns = [
    "voucher_id",
    "transaction_date",
    "transaction_type",
    "department_code",
    "partner_code",
    "evidence_type",
    "evidence_no",
    "description"
]

bulk_normal_journal["required_values_valid"] = (
    bulk_normal_journal[required_columns]
    .notna()
    .all(axis=1)
)

# 모든 검증 항목을 통과했는지 최종 판단
quality_check_columns = [
    "balance_valid",
    "total_amount_valid",
    "vat_valid",
    "partner_valid",
    "department_valid",
    "account_valid",
    "date_valid",
    "required_values_valid"
]

bulk_normal_journal["all_checks_passed"] = (
    bulk_normal_journal[quality_check_columns]
    .all(axis=1)
)

# 검증을 통과하지 못한 거래만 추출
invalid_generated_rows = bulk_normal_journal.loc[
    bulk_normal_journal["all_checks_passed"] == False
]

# 중복 식별자 검사
duplicate_voucher_count = (
    bulk_normal_journal["voucher_id"]
    .duplicated()
    .sum()
)

duplicate_evidence_count = (
    bulk_normal_journal["evidence_no"]
    .duplicated()
    .sum()
)

print("검증한 거래 수:", len(bulk_normal_journal))
print("검증 실패 거래 수:", len(invalid_generated_rows))
print("중복 전표번호 수:", duplicate_voucher_count)
print("중복 증빙번호 수:", duplicate_evidence_count)

# 검증 항목별 통과 건수 확인
quality_summary = pd.DataFrame(
    {
        "check_name": quality_check_columns,
        "passed_count": [
            int(bulk_normal_journal[column].sum())
            for column in quality_check_columns
        ]
    }
)

quality_summary

검증한 거래 수: 500
검증 실패 거래 수: 0
중복 전표번호 수: 0
중복 증빙번호 수: 0


,check_name,passed_count
0,balance_valid,500
1,total_amount_valid,500
2,vat_valid,500
3,partner_valid,500
4,department_valid,500
5,account_valid,500
6,date_valid,500
7,required_values_valid,500


## 5. 대규모 분개장에 테스트 오류 삽입

정상 거래 500건 중 서로 다른 50건을 선택한다.
10가지 오류 유형을 각각 5건씩 삽입하고 별도의 정답표를 생성한다.

In [6]:
# 정상 데이터 검증 과정에서 추가한 임시 열 목록
temporary_check_columns = [
    "debit_total_check",
    "credit_total_check",
    "balance_valid",
    "total_amount_valid",
    "vat_valid",
    "partner_valid",
    "department_valid",
    "account_valid",
    "date_valid",
    "required_values_valid",
    "all_checks_passed"
]

# 검증용 임시 열을 제외하고 실제 분개장 데이터만 복사
bulk_journal = bulk_normal_journal.drop(
    columns=temporary_check_columns
).copy()

# 삽입할 오류 유형과 오류가 발생하는 열 정의
error_definitions = [
    {
        "error_type": "존재하지 않는 계정과목",
        "column": "debit_account_1"
    },
    {
        "error_type": "존재하지 않는 거래처",
        "column": "partner_code"
    },
    {
        "error_type": "존재하지 않는 부서",
        "column": "department_code"
    },
    {
        "error_type": "차변·대변 불일치",
        "column": "debit_amount_1"
    },
    {
        "error_type": "부가세 계산 오류",
        "column": "vat_amount"
    },
    {
        "error_type": "필수값 누락",
        "column": "evidence_no"
    },
    {
        "error_type": "증빙번호 중복",
        "column": "evidence_no"
    },
    {
        "error_type": "회계기간 이탈",
        "column": "transaction_date"
    },
    {
        "error_type": "필수값 누락",
        "column": "description"
    },
    {
        "error_type": "전표 합계 불일치",
        "column": "total_amount"
    }
]

# 각 오류 유형을 5번씩 반복해 총 50개의 오류 목록 생성
error_assignments = []

for error_definition in error_definitions:
    for _ in range(5):
        error_assignments.append(
            error_definition.copy()
        )

# 오류 유형이 특정 구간에 몰리지 않도록 순서 섞기
np.random.shuffle(error_assignments)

# 0~9번 행은 중복 증빙의 비교 기준으로 보존
# 나머지 행 중 중복 없이 50개 행 선택
selected_error_indices = np.random.choice(
    bulk_journal.index[10:],
    size=error_transaction_count,
    replace=False
)

# 중복 증빙 오류에 사용할 정상 증빙번호
duplicate_evidence_anchor = bulk_journal.loc[
    0,
    "evidence_no"
]

# 삽입한 오류의 정답 정보를 저장할 빈 목록
bulk_expected_errors = []


# 선택된 50개 거래에 오류를 하나씩 삽입
for error_number, (
    row_index,
    error_assignment
) in enumerate(
    zip(
        selected_error_indices,
        error_assignments
    ),
    start=1
):
    error_type = error_assignment["error_type"]
    error_column = error_assignment["column"]

    # 수정 전 원본값 보관
    original_value = bulk_journal.loc[
        row_index,
        error_column
    ]

    # 오류 유형에 따라 데이터 수정
    if error_type == "존재하지 않는 계정과목":
        bulk_journal.loc[
            row_index,
            "debit_account_1"
        ] = 9999

    elif error_type == "존재하지 않는 거래처":
        bulk_journal.loc[
            row_index,
            "partner_code"
        ] = "V999"

    elif error_type == "존재하지 않는 부서":
        bulk_journal.loc[
            row_index,
            "department_code"
        ] = "D999"

    elif error_type == "차변·대변 불일치":
        bulk_journal.loc[
            row_index,
            "debit_amount_1"
        ] += 10_000

    elif error_type == "부가세 계산 오류":
        bulk_journal.loc[
            row_index,
            "vat_amount"
        ] += 10_000

    elif (
        error_type == "필수값 누락"
        and error_column == "evidence_no"
    ):
        bulk_journal.loc[
            row_index,
            "evidence_no"
        ] = None

    elif error_type == "증빙번호 중복":
        bulk_journal.loc[
            row_index,
            "evidence_no"
        ] = duplicate_evidence_anchor

    elif error_type == "회계기간 이탈":
        bulk_journal.loc[
            row_index,
            "transaction_date"
        ] = pd.Timestamp("2026-09-05")

    elif (
        error_type == "필수값 누락"
        and error_column == "description"
    ):
        bulk_journal.loc[
            row_index,
            "description"
        ] = None

    elif error_type == "전표 합계 불일치":
        bulk_journal.loc[
            row_index,
            "total_amount"
        ] -= 10_000

    # 삽입한 오류의 정답 정보 기록
    bulk_expected_errors.append(
        {
            "error_id": f"ERR{error_number:03d}",
            "voucher_id": bulk_journal.loc[
                row_index,
                "voucher_id"
            ],
            "column": error_column,
            "error_type": error_type,
            "original_value": original_value,
            "injected_value": bulk_journal.loc[
                row_index,
                error_column
            ]
        }
    )


# 오류 정답 목록을 데이터프레임으로 변환
bulk_expected_errors_df = pd.DataFrame(
    bulk_expected_errors
)

# 오류 유형별 삽입 건수 집계
bulk_error_summary = (
    bulk_expected_errors_df
    .groupby(
        [
            "error_type",
            "column"
        ]
    )
    .size()
    .reset_index(name="error_count")
)

print("전체 분개장 거래 수:", len(bulk_journal))
print(
    "오류가 삽입된 거래 수:",
    bulk_expected_errors_df["voucher_id"].nunique()
)
print("삽입한 전체 오류 수:", len(bulk_expected_errors_df))

bulk_error_summary

전체 분개장 거래 수: 500
오류가 삽입된 거래 수: 50
삽입한 전체 오류 수: 50


,error_type,column,error_count
0,부가세 계산 오류,vat_amount,5
1,전표 합계 불일치,total_amount,5
2,존재하지 않는 거래처,partner_code,5
3,존재하지 않는 계정과목,debit_account_1,5
4,존재하지 않는 부서,department_code,5
5,증빙번호 중복,evidence_no,5
6,차변·대변 불일치,debit_amount_1,5
7,필수값 누락,description,5
8,필수값 누락,evidence_no,5
9,회계기간 이탈,transaction_date,5


## 6. 대규모 분개장과 오류 정답표 저장

오류 50건이 포함된 분개장 500건을 검증 대상 파일로 저장한다.
오류가 없는 원본 데이터와 삽입한 오류 정답표도 비교·복구용으로 별도 저장한다.

In [7]:
# 검증용 임시 열을 제거한 정상 원본 데이터
bulk_normal_reference = bulk_normal_journal.drop(
    columns=temporary_check_columns
).copy()

# 대규모 오류 분개장 저장 경로
bulk_journal_path = (
    project_root
    / "data"
    / "raw"
    / "bulk_journal.xlsx"
)

# 오류 삽입 전 정상 원본 저장 경로
bulk_normal_reference_path = (
    project_root
    / "data"
    / "expected"
    / "bulk_normal_reference.xlsx"
)

# 삽입한 오류 정답표 저장 경로
bulk_expected_errors_path = (
    project_root
    / "data"
    / "expected"
    / "bulk_injected_errors.csv"
)

# 오류 유형별 요약 저장 경로
bulk_error_summary_path = (
    project_root
    / "data"
    / "expected"
    / "bulk_error_summary.csv"
)


# 오류 50건이 포함된 분개장 저장
bulk_journal.to_excel(
    bulk_journal_path,
    index=False,
    sheet_name="분개장"
)

# 오류 삽입 전 정상 분개장 저장
bulk_normal_reference.to_excel(
    bulk_normal_reference_path,
    index=False,
    sheet_name="정상_분개장"
)

# 오류별 정답 정보 저장
bulk_expected_errors_df.to_csv(
    bulk_expected_errors_path,
    index=False,
    encoding="utf-8-sig"
)

# 오류 유형별 삽입 건수 저장
bulk_error_summary.to_csv(
    bulk_error_summary_path,
    index=False,
    encoding="utf-8-sig"
)


# 저장된 파일을 다시 불러와 행 개수 확인
saved_bulk_journal = pd.read_excel(
    bulk_journal_path,
    sheet_name="분개장"
)

saved_expected_errors = pd.read_csv(
    bulk_expected_errors_path
)

print("대규모 오류 분개장 저장:", bulk_journal_path.exists())
print(
    "정상 원본 저장:",
    bulk_normal_reference_path.exists()
)
print(
    "오류 정답표 저장:",
    bulk_expected_errors_path.exists()
)
print(
    "오류 요약 저장:",
    bulk_error_summary_path.exists()
)

print()
print("다시 불러온 분개장 거래 수:", len(saved_bulk_journal))
print("다시 불러온 오류 정답 수:", len(saved_expected_errors))

대규모 오류 분개장 저장: True
정상 원본 저장: True
오류 정답표 저장: True
오류 요약 저장: True

다시 불러온 분개장 거래 수: 500
다시 불러온 오류 정답 수: 50
